<a href="https://colab.research.google.com/github/abdullahawan0043-glitch/Flyrank-machine-learning-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
"""
1. Unit of analysis: One row = one content page, belonging to one client,
summarized over one month-long window (month = 2026-03). The underlying
daily fact table has finer grain (one row per page per client per day),
which I aggregate up to page-level for this lane.

Table(s) used: fact_content_daily_performance (daily search/engagement
facts), joined with dim_content (page metadata) and dim_clients (to check
each client's gsc_data_start/ga4_data_start before trusting a window).

Time window: I iterate on month = 2026-03 (a mid-panel month), per the
warehouse guide's warning that the final month (2026-06) is the sealed
test month and must never be used to develop label logic.
"""

"\n1. Unit of analysis: One row = one content page, belonging to one client,\nsummarized over one month-long window (month = 2026-03). The underlying\ndaily fact table has finer grain (one row per page per client per day),\nwhich I aggregate up to page-level for this lane.\n\nTable(s) used: fact_content_daily_performance (daily search/engagement\nfacts), joined with dim_content (page metadata) and dim_clients (to check\neach client's gsc_data_start/ga4_data_start before trusting a window).\n\nTime window: I iterate on month = 2026-03 (a mid-panel month), per the\nwarehouse guide's warning that the final month (2026-06) is the sealed\ntest month and must never be used to develop label logic.\n"

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
"""
2. Field classification for this lane:

FEATURES (knowable before the decision moment):
- impressions, clicks, ctr, avg_position (gsc_avg_position) - search visibility
- sessions, engagement_rate, scroll_rate - engagement signals
- content_age_days, word_count - content metadata

LABEL / PROXY (the thing I would predict - built in later weeks):
- A future decline in impressions, measured in a window AFTER the feature
  window. Never used as a feature itself.

CONTEXT (for grouping/joining only, never fed to a model):
- content_hash_id, client_hash_id, report_date, url_hash_id

EXCLUDED (with reason):
- Raw URLs, raw queries, client names - scrambled/removed before release,
  and using them risks re-identifying private client data.
- FlyRank product decision fields (health_score, priority_score,
  action_type) - not shipped in this release; even if available, using
  them would just teach a model to copy FlyRank's existing rule instead
  of discovering real signal.
- The final month (2026-06) - deliberately excluded from iteration since
  it is the sealed test window.
"""

"\n2. Field classification for this lane:\n\nFEATURES (knowable before the decision moment):\n- impressions, clicks, ctr, avg_position (gsc_avg_position) - search visibility\n- sessions, engagement_rate, scroll_rate - engagement signals\n- content_age_days, word_count - content metadata\n\nLABEL / PROXY (the thing I would predict - built in later weeks):\n- A future decline in impressions, measured in a window AFTER the feature\n  window. Never used as a feature itself.\n\nCONTEXT (for grouping/joining only, never fed to a model):\n- content_hash_id, client_hash_id, report_date, url_hash_id\n\nEXCLUDED (with reason):\n- Raw URLs, raw queries, client names - scrambled/removed before release,\n  and using them risks re-identifying private client data.\n- FlyRank product decision fields (health_score, priority_score,\n  action_type) - not shipped in this release; even if available, using\n  them would just teach a model to copy FlyRank's existing rule instead\n  of discovering real signal

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
import duckdb

hf_token = "YOUR_HF_TOKEN_HERE"

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}');")
print("Connected.\n")

path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

grain_check = con.sql(f"""
SELECT client_hash_id, content_hash_id, report_date, COUNT(*) c
FROM read_parquet('{path}')
GROUP BY client_hash_id, content_hash_id, report_date
HAVING c > 1 LIMIT 5
""").df()
print("Grain check:"); print(grain_check)
print(f"Rows returned: {len(grain_check)} (0 means grain holds)\n")

counts_check = con.sql(f"""
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT client_hash_id) AS n_clients,
       COUNT(DISTINCT content_hash_id) AS n_content_items,
       MIN(report_date) AS min_date, MAX(report_date) AS max_date
FROM read_parquet('{path}')
""").df()
print("Row count and date span:"); print(counts_check); print()

availability_check = con.sql(f"""
SELECT COUNT(*) AS total_rows,
       SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4,
       SUM(CASE WHEN gsc_impressions IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_impressions
FROM read_parquet('{path}')
""").df()
print("Availability check:"); print(availability_check); print()

features_frame = con.sql(f"""
SELECT content_hash_id, client_hash_id,
       AVG(gsc_impressions) AS avg_impressions,
       AVG(gsc_clicks) AS avg_clicks,
       AVG(CASE WHEN gsc_impressions > 0 THEN gsc_clicks * 1.0 / gsc_impressions ELSE NULL END) AS avg_ctr,
       AVG(gsc_avg_position) AS avg_position,
       AVG(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions * 1.0 / NULLIF(ga4_sessions, 0) ELSE NULL END) AS avg_engagement_rate
FROM read_parquet('{path}')
GROUP BY content_hash_id, client_hash_id LIMIT 20
""").df()
print("Five-feature frame:"); print(features_frame)
print("""
Feature availability notes:
1. avg_impressions (gsc_impressions) - past search-visibility measurement, known before any future window.
2. avg_clicks (gsc_clicks) - observed past outcome, not derived from a future label.
3. avg_ctr - computed only from past clicks/impressions.
4. avg_position (gsc_avg_position) - past average ranking position, observed historically.
5. avg_engagement_rate - only used when ga4_data_available IS TRUE, computed from ga4_engaged_sessions/ga4_sessions.
""")

X_features = features_frame[["avg_impressions", "avg_clicks", "avg_ctr", "avg_position"]].fillna(0)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

y_fake = (X_features["avg_position"] > X_features["avg_position"].median()).astype(int)
model_honest = LogisticRegression(max_iter=1000).fit(X_features, y_fake)
honest_score = roc_auc_score(y_fake, model_honest.predict_proba(X_features)[:, 1])
print(f"Honest ROC-AUC: {honest_score:.3f}")

X_leaky = X_features.copy()
X_leaky["leaky_column"] = y_fake
model_leaky = LogisticRegression(max_iter=1000).fit(X_leaky, y_fake)
leaky_score = roc_auc_score(y_fake, model_leaky.predict_proba(X_leaky)[:, 1])
print(f"Leaky ROC-AUC (with label-derived column): {leaky_score:.3f}")
print("-> Score jumps toward perfect - this is the leakage trap.")

del X_leaky["leaky_column"]
print(f"Leaky column removed. Keeping honest score: {honest_score:.3f}")

Connected.



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check:
Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, c]
Index: []
Rows returned: 0 (0 means grain holds)

Row count and date span:
   total_rows  n_clients  n_content_items   min_date   max_date
0     9841378         55           331437 2026-03-01 2026-03-31



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Availability check:
   total_rows  rows_with_ga4  rows_with_impressions
0     9841378       413966.0              9841378.0



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Five-feature frame:
             content_hash_id           client_hash_id  avg_impressions  \
0   content_d0dff76c889de68f  client_62f4a7e64f5e0096         5.838710   
1   content_ac8663da7484669a  client_62f4a7e64f5e0096         1.096774   
2   content_39d7361b4945d504  client_62f4a7e64f5e0096         2.483871   
3   content_d49a012dcb924e31  client_62f4a7e64f5e0096        10.612903   
4   content_cec711b02f3bbde6  client_62f4a7e64f5e0096        19.419355   
5   content_614baf2af4330bd7  client_62f4a7e64f5e0096        24.903226   
6   content_ceaec531566ffcfc  client_62f4a7e64f5e0096         2.645161   
7   content_755d951187fcd70a  client_62f4a7e64f5e0096        59.935484   
8   content_7a7d3c7aa7cdfc5c  client_62f4a7e64f5e0096         0.516129   
9   content_225dc9235023be5f  client_62f4a7e64f5e0096        15.741935   
10  content_cdd114d71966c437  client_62f4a7e64f5e0096        60.903226   
11  content_50b73974582ead90  client_62f4a7e64f5e0096         3.516129   
12  content_7dbc09

In [ ]:
schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{path}') LIMIT 1").df()
print(schema.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
"""
4. Named limitation of this slice:

This month=2026-03 slice is an unbalanced panel: not every client has a
full history at this point. Some clients' gsc_data_start or ga4_data_start
falls after March 2026, so their rows may show ga4_data_available = FALSE
even though the page was active - this is "no tracking yet," not "no
engagement." Because of this, any feature or count built from this single
month may under-represent clients who onboarded later, and conclusions
here should be treated as directional for this slice, not as a
site-wide or client-wide guarantee.
"""

'\n4. Named limitation of this slice:\n\nThis month=2026-03 slice is an unbalanced panel: not every client has a\nfull history at this point. Some clients\' gsc_data_start or ga4_data_start\nfalls after March 2026, so their rows may show ga4_data_available = FALSE\neven though the page was active - this is "no tracking yet," not "no\nengagement." Because of this, any feature or count built from this single\nmonth may under-represent clients who onboarded later, and conclusions\nhere should be treated as directional for this slice, not as a\nsite-wide or client-wide guarantee.\n'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.